# Laboratorio 2 - Minería de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
import seaborn as sb
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as sch
import skfuzzy as fuzz
import pylab
import sklearn.mixture as mixture
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import pyclustertend 
import random
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import KMeans, AgglomerativeClustering


In [3]:
#importar csv
df = pd.read_csv('movies_2026.csv', encoding='latin-1')
pd.options.display.float_format = '{:,.0f}'.format

## 1. Clustering

In [4]:
# 1.1. Preprocesamiento del dataset
vars_cluster = [
    'budget',
    'revenue',
    'popularity',
    'runtime',
    'voteAvg',
    'voteCount',
    'actorsAmount',
    'castWomenAmount',
    'castMenAmount',
    'genresAmount',
    'productionCoAmount',
    'productionCountriesAmount'
]

#Escalar datos
X = df[vars_cluster].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [5]:
# 1.2. Estadístico de Hopkings y la VAT 
def hopkins(X):
    from sklearn.neighbors import NearestNeighbors
    import random
    
    d = X.shape[1]
    n = X.shape[0]
    m = int(0.1 * n)
    
    nbrs = NearestNeighbors(n_neighbors=1).fit(X)
    
    rand_X = np.random.uniform(np.min(X, axis=0),
                               np.max(X, axis=0),
                               (m,d))
    
    u_dist = []
    w_dist = []
    
    for j in range(m):
        u_dist.append(nbrs.kneighbors([rand_X[j]], 2, return_distance=True)[0][0][1])
        w_dist.append(nbrs.kneighbors([X[np.random.randint(0,n)]], 2, return_distance=True)[0][0][1])
    
    H = sum(u_dist) / (sum(u_dist) + sum(w_dist))
    return H

hopkins(X_scaled)

np.float64(0.9894721332505376)

In [6]:
#VAT
X_sample = X_scaled[np.random.choice(X_scaled.shape[0], 1500, replace=False)]
dist_matrix = squareform(pdist(X_sample))

def VAT(D):
    N = D.shape[0]
    J = list(range(N))
    
    I = []
    
    # Paso inicial
    y = np.argmax(np.sum(D, axis=1))
    I.append(y)
    J.remove(y)
    
    while J:
        j = max(J, key=lambda j: min([D[j, i] for i in I]))
        I.append(j)
        J.remove(j)
    
    D_reordered = D[np.ix_(I, I)]
    return D_reordered


D_vat = VAT(dist_matrix)

plt.figure(figsize=(8,8))
plt.imshow(D_vat, cmap='gray')
plt.title("VAT")
plt.colorbar()
plt.show()

KeyboardInterrupt: 

In [ ]:
# 1.3. Número de grupos a formar

inertia = []
K = range(2,10)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.plot(K, inertia, marker='o')
plt.title("Método del Codo")
plt.show()

In [ ]:
# 1.4. Algoritmos k-medias y clustering jerárquico
##k-means
kmeans = KMeans(n_clusters=4, random_state=42)
labels_k = kmeans.fit_prediction(X_scaled)

##Jerárquico
hc = AgglomerativeClustering(n_clusters = 4)
labels_h = hc.fit_predict(X_scaled)

In [ ]:
# 1.5. Calidad del agrupamiento
print("KMeans:", silhouette_score(X_scaled, labels_k))
print("Jerárquico:", silhouette_score(X_scaled, labels_h))

In [ ]:
# 1.6.  Interprete los grupos 
df_cluster = df.loc[X.index].copy()
df_cluster['Cluster'] = labels_k

df_cluster.groupby('Cluster')[vars_cluster].mean()

## 2. Reglas de Asociación

In [ ]:
# 

## 3. Análisis de Componentes Principales 

In [ ]:
# 3.2. conveniente hacer un Análisis de Componentes Principales.

In [ ]:
# 3.3. Análisis de componentes principales

## 4. Otros algoritmos de aprendizaje no supervisado  